Imports and global settings

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# LightGBM is used for the LambdaMART-style ranking model.
# If this import fails, install it with:
# conda install -c conda-forge lightgbm
# or:
# pip install lightgbm
import lightgbm as lgb

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42

Load train and test data

In [2]:
train_path = "data/training_set_VU_DM.csv"
test_path = "data/test_set_VU_DM.csv"

train_raw = pd.read_csv(train_path, na_values=["NULL"])
test_raw = pd.read_csv(test_path, na_values=["NULL"])

print("Train shape:", train_raw.shape)
print("Test shape:", test_raw.shape)

display(train_raw.head())
display(test_raw.head())

Train shape: (4958347, 54)
Test shape: (4959183, 50)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score1,prop_location_score2,prop_log_historical_price,position,price_usd,promotion_flag,srch_destination_id,srch_length_of_stay,srch_booking_window,srch_adults_count,srch_children_count,srch_room_count,srch_saturday_night_bool,srch_query_affinity_score,orig_destination_distance,random_bool,comp1_rate,comp1_inv,comp1_rate_percent_diff,comp2_rate,comp2_inv,comp2_rate_percent_diff,comp3_rate,comp3_inv,comp3_rate_percent_diff,comp4_rate,comp4_inv,comp4_rate_percent_diff,comp5_rate,comp5_inv,comp5_rate_percent_diff,comp6_rate,comp6_inv,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,1,2.83,0.0438,4.95,27,104.77,0,23246,1,0,4,0,1,1,NaN,NaN,1,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,1,2.20,0.0149,5.03,26,170.74,0,23246,1,0,4,0,1,1,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,1,2.20,0.0245,4.92,21,179.80,0,23246,1,0,4,0,1,1,NaN,NaN,1,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,1,2.83,0.0125,4.39,34,602.77,0,23246,1,0,4,0,1,1,NaN,NaN,1,NaN,NaN,NaN,-1.0,0.0,5.0,-1.0,0.0,5.0,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,1,2.64,0.1241,4.93,4,143.58,0,23246,1,0,4,0,1,1,NaN,NaN,1,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score1,prop_location_score2,prop_log_historical_price,price_usd,promotion_flag,srch_destination_id,srch_length_of_stay,srch_booking_window,srch_adults_count,srch_children_count,srch_room_count,srch_saturday_night_bool,srch_query_affinity_score,orig_destination_distance,random_bool,comp1_rate,comp1_inv,comp1_rate_percent_diff,comp2_rate,comp2_inv,comp2_rate_percent_diff,comp3_rate,comp3_inv,comp3_rate_percent_diff,comp4_rate,comp4_inv,comp4_rate_percent_diff,comp5_rate,comp5_inv,comp5_rate_percent_diff,comp6_rate,comp6_inv,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff
0,1,2013-02-02 15:27:40,24,216,NaN,NaN,219,3180,3,4.5,1,2.94,0.0691,5.03,119.0,0,19222,1,10,2,0,1,0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2013-02-02 15:27:40,24,216,NaN,NaN,219,5543,3,4.5,1,2.64,0.0843,4.93,118.0,0,19222,1,10,2,0,1,0,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2013-02-02 15:27:40,24,216,NaN,NaN,219,14142,2,3.5,1,2.71,0.0556,4.16,49.0,0,19222,1,10,2,0,1,0,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2013-02-02 15:27:40,24,216,NaN,NaN,219,22393,3,4.5,1,2.40,0.0561,5.03,143.0,0,19222,1,10,2,0,1,0,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,2013-02-02 15:27:40,24,216,NaN,NaN,219,24194,3,4.5,1,2.94,0.2090,4.72,79.0,0,19222,1,10,2,0,1,0,NaN,NaN,0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Create ranking target and define columns that cannot be model features

In [3]:
# Relevance follows the official NDCG logic:
# booking = 5, click-only = 1, ignored = 0
train_raw["relevance"] = np.where(
    train_raw["booking_bool"] == 1, 5,
    np.where(train_raw["click_bool"] == 1, 1, 0)
).astype(np.int8)

# Columns that must not be used as input features:
# - click_bool and booking_bool are target variables
# - gross_bookings_usd is target-related and mostly only known after booking
# - position is only available in train and captures Expedia's original ranking
# - relevance is the constructed target
LEAK_COLS = [
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

print(train_raw["relevance"].value_counts(normalize=True).sort_index())

relevance
0    0.955251
1    0.016838
5    0.027911
Name: proportion, dtype: float64


Split train into train/validation by complete search groups

In [4]:
# We split by srch_id, not by rows.
# This prevents hotels from the same search appearing in both train and validation.
unique_searches = train_raw["srch_id"].unique()

train_srch_ids, val_srch_ids = train_test_split(
    unique_searches,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_fold = train_raw[train_raw["srch_id"].isin(train_srch_ids)].copy()
val_fold = train_raw[train_raw["srch_id"].isin(val_srch_ids)].copy()

print("Train fold shape:", train_fold.shape)
print("Validation fold shape:", val_fold.shape)
print("Train searches:", train_fold["srch_id"].nunique())
print("Validation searches:", val_fold["srch_id"].nunique())

Train fold shape: (3966682, 55)
Validation fold shape: (991665, 55)
Train searches: 159836
Validation searches: 39959


Memory reduction helper -> Check if we still want this later on!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [5]:
def reduce_memory_usage(df):
    """
    Downcasts numeric columns to reduce memory usage.
    This is useful because the Expedia dataset is large.
    """
    for col in df.columns:
        col_type = df[col].dtype

        if col_type == "object":
            continue

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

    return df

Fit preprocessing values on training fold only

In [6]:
def fit_preprocessing_params(df):
    """
    Computes imputation values and caps using only the training fold.
    These values are then reused for validation and test, preventing leakage.
    """

    params = {}

    # Imputation values
    params["visitor_hist_starrating_fill"] = -1
    params["visitor_hist_adr_usd_fill"] = -1

    # srch_query_affinity_score is a negative log probability.
    # Missing gets a value slightly below the minimum observed value.
    params["srch_query_affinity_score_fill"] = df["srch_query_affinity_score"].min(skipna=True) - 1

    params["orig_destination_distance_fill"] = df["orig_destination_distance"].median()
    params["prop_location_score2_fill"] = df["prop_location_score2"].median()
    params["prop_review_score_fill"] = df["prop_review_score"].median()

    # Price cap based on training fold only.
    # We cap extreme outliers but still keep price information.
    params["price_usd_cap"] = df["price_usd"].quantile(0.999)

    # Global fallback values for aggregate features
    params["global_price_mean"] = df["price_usd"].mean()
    params["global_price_median"] = df["price_usd"].median()
    params["global_price_std"] = df["price_usd"].std()
    params["global_star_mean"] = df["prop_starrating"].mean()
    params["global_review_mean"] = df["prop_review_score"].mean()
    params["global_loc1_mean"] = df["prop_location_score1"].mean()
    params["global_loc2_mean"] = df["prop_location_score2"].mean()
    params["global_promotion_mean"] = df["promotion_flag"].mean()

    return params


preprocess_params = fit_preprocessing_params(train_fold)
preprocess_params

{'visitor_hist_starrating_fill': -1,
 'visitor_hist_adr_usd_fill': -1,
 'srch_query_affinity_score_fill': np.float64(-327.5675),
 'orig_destination_distance_fill': np.float64(386.42),
 'prop_location_score2_fill': np.float64(0.069),
 'prop_review_score_fill': np.float64(4.0),
 'price_usd_cap': np.float64(2039.5573500000871),
 'global_price_mean': np.float64(271.897185304998),
 'global_price_median': np.float64(122.0),
 'global_price_std': np.float64(17854.751933387954),
 'global_star_mean': np.float64(3.180235521778655),
 'global_review_mean': np.float64(3.7781418037476286),
 'global_loc1_mean': np.float64(2.8729895892839403),
 'global_loc2_mean': np.float64(0.13032077072440965),
 'global_promotion_mean': np.float64(0.21583101443473413)}

Basic cleaning, missing indicators, imputations, and simple transformations

In [7]:
def add_basic_features(df, params, is_train=True):
    """
    Adds basic non-target features:
    - missing indicators
    - missing-value imputations
    - price transformations
    - review/star zero indicators
    - trip/user context features
    - date/time features
    """

    df = df.copy()

    # -------------------------
    # Missing indicators
    # -------------------------
    df["visitor_hist_starrating_missing"] = df["visitor_hist_starrating"].isna().astype(np.int8)
    df["visitor_hist_adr_usd_missing"] = df["visitor_hist_adr_usd"].isna().astype(np.int8)
    df["srch_query_affinity_score_missing"] = df["srch_query_affinity_score"].isna().astype(np.int8)
    df["orig_destination_distance_missing"] = df["orig_destination_distance"].isna().astype(np.int8)
    df["prop_location_score2_missing"] = df["prop_location_score2"].isna().astype(np.int8)
    df["prop_review_score_missing"] = df["prop_review_score"].isna().astype(np.int8)

    # -------------------------
    # Special zero indicators
    # -------------------------
    # prop_review_score = 0 means no reviews.
    # prop_starrating = 0 means unknown/no stars/not publicized.
    df["prop_review_score_is_zero"] = (df["prop_review_score"] == 0).astype(np.int8)
    df["prop_starrating_is_zero"] = (df["prop_starrating"] == 0).astype(np.int8)

    # -------------------------
    # Impute missing values
    # -------------------------
    df["visitor_hist_starrating"] = df["visitor_hist_starrating"].fillna(params["visitor_hist_starrating_fill"])
    df["visitor_hist_adr_usd"] = df["visitor_hist_adr_usd"].fillna(params["visitor_hist_adr_usd_fill"])
    df["srch_query_affinity_score"] = df["srch_query_affinity_score"].fillna(params["srch_query_affinity_score_fill"])
    df["orig_destination_distance"] = df["orig_destination_distance"].fillna(params["orig_destination_distance_fill"])
    df["prop_location_score2"] = df["prop_location_score2"].fillna(params["prop_location_score2_fill"])
    df["prop_review_score"] = df["prop_review_score"].fillna(params["prop_review_score_fill"])

    # -------------------------
    # Price transformations
    # -------------------------
    df["price_usd_capped"] = df["price_usd"].clip(upper=params["price_usd_cap"])
    df["log_price_usd"] = np.log1p(df["price_usd"])
    df["log_price_usd_capped"] = np.log1p(df["price_usd_capped"])

    # -------------------------
    # Distance transformation
    # -------------------------
    df["log_orig_destination_distance"] = np.log1p(df["orig_destination_distance"])

    # -------------------------
    # Trip/user context features
    # -------------------------
    df["total_guests"] = df["srch_adults_count"] + df["srch_children_count"]
    df["children_present"] = (df["srch_children_count"] > 0).astype(np.int8)
    df["multi_room"] = (df["srch_room_count"] > 1).astype(np.int8)
    df["family_search"] = ((df["srch_children_count"] > 0) | (df["srch_room_count"] > 1)).astype(np.int8)

    df["domestic_search"] = (
        df["visitor_location_country_id"] == df["prop_country_id"]
    ).astype(np.int8)

    df["log_booking_window"] = np.log1p(df["srch_booking_window"])
    df["log_length_of_stay"] = np.log1p(df["srch_length_of_stay"])

    df["last_minute_booking"] = (df["srch_booking_window"] <= 1).astype(np.int8)
    df["long_booking_window"] = (df["srch_booking_window"] >= 150).astype(np.int8)
    df["long_stay"] = (df["srch_length_of_stay"] >= 7).astype(np.int8)

    # -------------------------
    # Star/review non-linear indicators
    # -------------------------
    df["is_4_star"] = (df["prop_starrating"] == 4).astype(np.int8)
    df["is_5_star"] = (df["prop_starrating"] == 5).astype(np.int8)
    df["high_review_score"] = (df["prop_review_score"] >= 4.0).astype(np.int8)

    # -------------------------
    # Date/time features
    # -------------------------
    df["date_time"] = pd.to_datetime(df["date_time"])

    df["search_month"] = df["date_time"].dt.month.astype(np.int8)
    df["search_dayofweek"] = df["date_time"].dt.dayofweek.astype(np.int8)
    df["search_hour"] = df["date_time"].dt.hour.astype(np.int8)
    df["search_is_weekend"] = df["search_dayofweek"].isin([5, 6]).astype(np.int8)

    return reduce_memory_usage(df)

Competitor missingness and aggregate competitor features

In [8]:
comp_rate_cols = [f"comp{i}_rate" for i in range(1, 9)]
comp_inv_cols = [f"comp{i}_inv" for i in range(1, 9)]
comp_pct_cols = [f"comp{i}_rate_percent_diff" for i in range(1, 9)]

def add_competitor_features(df):
    """
    Adds competitor features.
    The raw competitor columns are very sparse, so we summarize them with counts.
    """

    df = df.copy()

    # -------------------------
    # Missing-count features before filling
    # -------------------------
    df["comp_rate_missing_count"] = df[comp_rate_cols].isna().sum(axis=1).astype(np.int8)
    df["comp_inv_missing_count"] = df[comp_inv_cols].isna().sum(axis=1).astype(np.int8)
    df["comp_pct_missing_count"] = df[comp_pct_cols].isna().sum(axis=1).astype(np.int8)

    # -------------------------
    # Any competitor data available?
    # -------------------------
    df["comp_has_any_rate_data"] = (df["comp_rate_missing_count"] < 8).astype(np.int8)
    df["comp_has_any_inv_data"] = (df["comp_inv_missing_count"] < 8).astype(np.int8)
    df["comp_has_any_pct_data"] = (df["comp_pct_missing_count"] < 8).astype(np.int8)

    # -------------------------
    # Aggregate features before filling
    # comp_rate:
    #   +1 means Expedia cheaper
    #    0 means same price
    #   -1 means Expedia more expensive
    # comp_inv:
    #   +1 means competitor unavailable
    #    0 means competitor available
    # -------------------------
    df["comp_expedia_cheaper_count"] = df[comp_rate_cols].eq(1).sum(axis=1).astype(np.int8)
    df["comp_expedia_more_expensive_count"] = df[comp_rate_cols].eq(-1).sum(axis=1).astype(np.int8)
    df["comp_same_price_count"] = df[comp_rate_cols].eq(0).sum(axis=1).astype(np.int8)

    df["comp_unavailable_count"] = df[comp_inv_cols].eq(1).sum(axis=1).astype(np.int8)
    df["comp_available_count"] = df[comp_inv_cols].eq(0).sum(axis=1).astype(np.int8)

    # -------------------------
    # Missing flags and fill raw competitor columns
    # -------------------------
    for i in range(1, 9):
        rate_col = f"comp{i}_rate"
        inv_col = f"comp{i}_inv"
        pct_col = f"comp{i}_rate_percent_diff"

        df[f"{rate_col}_missing"] = df[rate_col].isna().astype(np.int8)
        df[f"{inv_col}_missing"] = df[inv_col].isna().astype(np.int8)
        df[f"{pct_col}_missing"] = df[pct_col].isna().astype(np.int8)

        # -2 means no competitor data.
        # This is outside the normal values of -1, 0, 1.
        df[rate_col] = df[rate_col].fillna(-2)
        df[inv_col] = df[inv_col].fillna(-2)

        # For percent difference, missing means no known difference.
        # We already captured missingness separately, so fill with 0.
        df[pct_col] = df[pct_col].fillna(0)

    df["comp_no_rate_data_count"] = df[comp_rate_cols].eq(-2).sum(axis=1).astype(np.int8)
    df["comp_no_inv_data_count"] = df[comp_inv_cols].eq(-2).sum(axis=1).astype(np.int8)

    return reduce_memory_usage(df)

Within-search relative features

In [9]:

def add_within_search_features(df):
    """
    Adds features comparing each hotel to the other hotels in the same search.
    These features are essential because the task is to rank hotels within srch_id.
    """

    df = df.copy()

    group = df.groupby("srch_id")

    # -------------------------
    # Number of hotels in the search
    # -------------------------
    df["hotels_in_search"] = group["prop_id"].transform("count")

    # -------------------------
    # Price relative to search
    # -------------------------
    df["search_price_mean"] = group["price_usd_capped"].transform("mean")
    df["search_price_median"] = group["price_usd_capped"].transform("median")
    df["search_price_min"] = group["price_usd_capped"].transform("min")
    df["search_price_max"] = group["price_usd_capped"].transform("max")

    df["price_diff_from_search_mean"] = df["price_usd_capped"] - df["search_price_mean"]
    df["price_diff_from_search_median"] = df["price_usd_capped"] - df["search_price_median"]
    df["price_diff_from_search_min"] = df["price_usd_capped"] - df["search_price_min"]

    df["price_ratio_to_search_mean"] = df["price_usd_capped"] / (df["search_price_mean"] + 1e-6)
    df["price_ratio_to_search_median"] = df["price_usd_capped"] / (df["search_price_median"] + 1e-6)

    # Lower price gets rank 1.
    df["price_rank_in_search"] = group["price_usd_capped"].rank(method="average", ascending=True)
    df["price_pct_rank_in_search"] = group["price_usd_capped"].rank(method="average", pct=True, ascending=True)

    # -------------------------
    # Log price relative to search
    # -------------------------
    df["search_log_price_mean"] = group["log_price_usd_capped"].transform("mean")
    df["log_price_diff_from_search_mean"] = df["log_price_usd_capped"] - df["search_log_price_mean"]

    # -------------------------
    # Star rating relative to search
    # -------------------------
    df["search_star_mean"] = group["prop_starrating"].transform("mean")
    df["search_star_max"] = group["prop_starrating"].transform("max")

    df["star_diff_from_search_mean"] = df["prop_starrating"] - df["search_star_mean"]
    df["star_diff_from_search_max"] = df["prop_starrating"] - df["search_star_max"]

    # Higher star rating gets rank 1.
    df["star_rank_in_search"] = group["prop_starrating"].rank(method="average", ascending=False)
    df["star_pct_rank_in_search"] = group["prop_starrating"].rank(method="average", pct=True, ascending=False)

    # -------------------------
    # Review score relative to search
    # -------------------------
    df["search_review_mean"] = group["prop_review_score"].transform("mean")
    df["search_review_max"] = group["prop_review_score"].transform("max")

    df["review_diff_from_search_mean"] = df["prop_review_score"] - df["search_review_mean"]
    df["review_diff_from_search_max"] = df["prop_review_score"] - df["search_review_max"]

    # Higher review gets rank 1.
    df["review_rank_in_search"] = group["prop_review_score"].rank(method="average", ascending=False)
    df["review_pct_rank_in_search"] = group["prop_review_score"].rank(method="average", pct=True, ascending=False)

    # -------------------------
    # Location score relative to search
    # -------------------------
    df["search_location1_mean"] = group["prop_location_score1"].transform("mean")
    df["search_location2_mean"] = group["prop_location_score2"].transform("mean")

    df["location1_diff_from_search_mean"] = df["prop_location_score1"] - df["search_location1_mean"]
    df["location2_diff_from_search_mean"] = df["prop_location_score2"] - df["search_location2_mean"]

    # Higher location score gets rank 1.
    df["location2_rank_in_search"] = group["prop_location_score2"].rank(method="average", ascending=False)
    df["location2_pct_rank_in_search"] = group["prop_location_score2"].rank(method="average", pct=True, ascending=False)

    # -------------------------
    # Promotion context
    # -------------------------
    df["search_promotion_rate"] = group["promotion_flag"].transform("mean")
    df["promotion_above_search_avg"] = df["promotion_flag"] - df["search_promotion_rate"]

    # -------------------------
    # Simple interaction features
    # -------------------------
    df["promotion_x_price_ratio"] = df["promotion_flag"] * df["price_ratio_to_search_mean"]
    df["brand_x_review"] = df["prop_brand_bool"] * df["prop_review_score"]
    df["star_x_review"] = df["prop_starrating"] * df["prop_review_score"]
    df["value_score"] = df["prop_starrating"] / (df["price_ratio_to_search_mean"] + 1e-6)

    return reduce_memory_usage(df)

Fit and apply non-target property/destination aggregate features

In [10]:
def fit_aggregate_features(df):
    """
    Fits safe aggregate features using non-target columns only.
    These are fitted on the training fold and merged into validation/test.
    """

    df = df.copy()

    # Make sure price transformations exist before fitting aggregates
    if "price_usd_capped" not in df.columns:
        raise ValueError("Run add_basic_features before fit_aggregate_features.")

    # -------------------------
    # Property-level aggregates
    # -------------------------
    prop_agg = df.groupby("prop_id").agg(
        prop_id_count=("prop_id", "size"),
        prop_id_mean_price=("price_usd_capped", "mean"),
        prop_id_median_price=("price_usd_capped", "median"),
        prop_id_std_price=("price_usd_capped", "std"),
        prop_id_mean_log_price=("log_price_usd_capped", "mean"),
        prop_id_mean_starrating=("prop_starrating", "mean"),
        prop_id_mean_review_score=("prop_review_score", "mean"),
        prop_id_mean_location_score1=("prop_location_score1", "mean"),
        prop_id_mean_location_score2=("prop_location_score2", "mean"),
        prop_id_mean_promotion_flag=("promotion_flag", "mean")
    ).reset_index()

    # -------------------------
    # Destination-level aggregates
    # -------------------------
    dest_agg = df.groupby("srch_destination_id").agg(
        dest_id_count=("srch_destination_id", "size"),
        dest_id_mean_price=("price_usd_capped", "mean"),
        dest_id_median_price=("price_usd_capped", "median"),
        dest_id_mean_starrating=("prop_starrating", "mean"),
        dest_id_mean_review_score=("prop_review_score", "mean"),
        dest_id_mean_location_score2=("prop_location_score2", "mean")
    ).reset_index()

    # -------------------------
    # Property-destination appearance count
    # This captures how often a property appears for a destination.
    # It does not use click/booking labels, so it is safer than target rates.
    # -------------------------
    prop_dest_agg = df.groupby(["prop_id", "srch_destination_id"]).agg(
        prop_dest_count=("prop_id", "size")
    ).reset_index()

    return prop_agg, dest_agg, prop_dest_agg


def add_aggregate_features(df, prop_agg, dest_agg, prop_dest_agg):
    """
    Merges fitted aggregate features into a dataframe.
    Missing aggregate values occur for unseen properties/destinations and are filled later.
    """

    df = df.copy()

    df = df.merge(prop_agg, on="prop_id", how="left")
    df = df.merge(dest_agg, on="srch_destination_id", how="left")
    df = df.merge(prop_dest_agg, on=["prop_id", "srch_destination_id"], how="left")

    return reduce_memory_usage(df)


def fill_aggregate_missing_values(df):
    """
    Fills aggregate features for unseen properties/destinations.
    Uses medians from the current dataframe after merging.
    """

    df = df.copy()

    agg_cols = [
        col for col in df.columns
        if col.startswith("prop_id_") or col.startswith("dest_id_") or col.startswith("prop_dest_")
    ]

    for col in agg_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return reduce_memory_usage(df)

Full feature preparation wrapper

In [11]:
def prepare_features(df, params, prop_agg=None, dest_agg=None, prop_dest_agg=None):
    """
    Applies the complete feature-engineering pipeline to a dataframe.
    """

    df = add_basic_features(df, params)
    df = add_competitor_features(df)
    df = add_within_search_features(df)

    if prop_agg is not None and dest_agg is not None and prop_dest_agg is not None:
        df = add_aggregate_features(df, prop_agg, dest_agg, prop_dest_agg)
        df = fill_aggregate_missing_values(df)

    df = reduce_memory_usage(df)

    return df

Prepare train and validation features

In [12]:
# Step 1: Basic, competitor, and within-search features
train_feat_no_agg = prepare_features(train_fold, preprocess_params)
val_feat_no_agg = prepare_features(val_fold, preprocess_params)

# Step 2: Fit non-target aggregates on training fold only
prop_agg, dest_agg, prop_dest_agg = fit_aggregate_features(train_feat_no_agg)

# Step 3: Merge aggregates into train and validation
train_feat = add_aggregate_features(train_feat_no_agg, prop_agg, dest_agg, prop_dest_agg)
val_feat = add_aggregate_features(val_feat_no_agg, prop_agg, dest_agg, prop_dest_agg)

train_feat = fill_aggregate_missing_values(train_feat)
val_feat = fill_aggregate_missing_values(val_feat)

print("Prepared train shape:", train_feat.shape)
print("Prepared validation shape:", val_feat.shape)

display(train_feat.head())

Prepared train shape: (3966682, 176)
Prepared validation shape: (991665, 176)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score1,prop_location_score2,prop_log_historical_price,position,price_usd,promotion_flag,srch_destination_id,srch_length_of_stay,srch_booking_window,srch_adults_count,srch_children_count,srch_room_count,srch_saturday_night_bool,srch_query_affinity_score,orig_destination_distance,random_bool,comp1_rate,comp1_inv,comp1_rate_percent_diff,comp2_rate,comp2_inv,comp2_rate_percent_diff,comp3_rate,comp3_inv,comp3_rate_percent_diff,comp4_rate,comp4_inv,comp4_rate_percent_diff,comp5_rate,comp5_inv,comp5_rate_percent_diff,comp6_rate,comp6_inv,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool,relevance,visitor_hist_starrating_missing,visitor_hist_adr_usd_missing,srch_query_affinity_score_missing,orig_destination_distance_missing,prop_location_score2_missing,prop_review_score_missing,prop_review_score_is_zero,prop_starrating_is_zero,price_usd_capped,log_price_usd,log_price_usd_capped,log_orig_destination_distance,total_guests,children_present,multi_room,family_search,domestic_search,log_booking_window,log_length_of_stay,last_minute_booking,long_booking_window,long_stay,is_4_star,is_5_star,high_review_score,search_month,search_dayofweek,search_hour,search_is_weekend,comp_rate_missing_count,comp_inv_missing_count,comp_pct_missing_count,comp_has_any_rate_data,comp_has_any_inv_data,comp_has_any_pct_data,comp_expedia_cheaper_count,comp_expedia_more_expensive_count,comp_same_price_count,comp_unavailable_count,comp_available_count,comp1_rate_missing,comp1_inv_missing,comp1_rate_percent_diff_missing,comp2_rate_missing,comp2_inv_missing,comp2_rate_percent_diff_missing,comp3_rate_missing,comp3_inv_missing,comp3_rate_percent_diff_missing,comp4_rate_missing,comp4_inv_missing,comp4_rate_percent_diff_missing,comp5_rate_missing,comp5_inv_missing,comp5_rate_percent_diff_missing,comp6_rate_missing,comp6_inv_missing,comp6_rate_percent_diff_missing,comp7_rate_missing,comp7_inv_missing,comp7_rate_percent_diff_missing,comp8_rate_missing,comp8_inv_missing,comp8_rate_percent_diff_missing,comp_no_rate_data_count,comp_no_inv_data_count,hotels_in_search,search_price_mean,search_price_median,search_price_min,search_price_max,price_diff_from_search_mean,price_diff_from_search_median,price_diff_from_search_min,price_ratio_to_search_mean,price_ratio_to_search_median,price_rank_in_search,price_pct_rank_in_search,search_log_price_mean,log_price_diff_from_search_mean,search_star_mean,search_star_max,star_diff_from_search_mean,star_diff_from_search_max,star_rank_in_search,star_pct_rank_in_search,search_review_mean,search_review_max,review_diff_from_search_mean,review_diff_from_search_max,review_rank_in_search,review_pct_rank_in_search,search_location1_mean,search_location2_mean,location1_diff_from_search_mean,location2_diff_from_search_mean,location2_rank_in_search,location2_pct_rank_in_search,search_promotion_rate,promotion_above_search_avg,promotion_x_price_ratio,brand_x_review,star_x_review,value_score,prop_id_count,prop_id_mean_price,prop_id_median_price,prop_id_std_price,prop_id_mean_log_price,prop_id_mean_starrating,prop_id_mean_review_score,prop_id_mean_location_score1,prop_id_mean_location_score2,prop_id_mean_promotion_flag,dest_id_count,dest_id_mean_price,dest_id_median_price,dest_id_mean_starrating,dest_id_mean_review_score,dest_id_mean_location_score2,prop_dest_count
0,1,2013-04-04 08:32:15,12,187,-1.0,-1.0,219,893,3,3.5,1,2.83,0.0438,4.95,27,104.77,0,23246,1,0,4,0,1,1,-327.567505,386.420013,1,-2.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,-2.0,-2.0,0.0,0.0,0.0,0.0,0,NaN,0,0,1,1,1,1,0,0,0,0,104.769997,4.661267,4.661267,5.959509,4,0,0,0,0,0.0,0.693147,1,0,0,0,0,0,4,3,8,0,4,4,8,1,1,0,0,0,4,0,4,1,1,1,0,0,1,0,0,1,1,1,1,0,0,1,1,1,1,1,1,1,0,0,1,4,4,

Build final feature list

In [13]:
# These columns are not used as model inputs.
# srch_id and prop_id are kept separately for grouping/submission.
NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

# Keep only numeric columns as model features.
feature_cols = [
    col for col in train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(train_feat[col])
]

# Make sure validation has exactly the same features.
missing_in_val = set(feature_cols) - set(val_feat.columns)
extra_in_val = set(val_feat.columns) - set(train_feat.columns)

print("Number of features:", len(feature_cols))
print("Missing in validation:", missing_in_val)
print("Extra in validation:", len(extra_in_val))

print(feature_cols[:50])

Number of features: 168
Missing in validation: set()
Extra in validation: 0
['site_id', 'visitor_location_country_id', 'visitor_hist_starrating', 'visitor_hist_adr_usd', 'prop_country_id', 'prop_starrating', 'prop_review_score', 'prop_brand_bool', 'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price', 'price_usd', 'promotion_flag', 'srch_destination_id', 'srch_length_of_stay', 'srch_booking_window', 'srch_adults_count', 'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool', 'srch_query_affinity_score', 'orig_destination_distance', 'random_bool', 'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff', 'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff', 'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff', 'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff', 'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff', 'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff', 'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff', 'comp8_rate', 'comp8_inv', 'comp8

Sort by srch_id and prepare X, y, and group sizes for ranking model

In [14]:
# LightGBM ranker needs rows sorted by query/search group.
train_feat = train_feat.sort_values("srch_id").reset_index(drop=True)
val_feat = val_feat.sort_values("srch_id").reset_index(drop=True)

X_train = train_feat[feature_cols]
y_train = train_feat["relevance"].astype(int)

X_val = val_feat[feature_cols]
y_val = val_feat["relevance"].astype(int)

group_train = train_feat.groupby("srch_id").size().to_numpy()
group_val = val_feat.groupby("srch_id").size().to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Number of train groups:", len(group_train))
print("Number of validation groups:", len(group_val))
print("First 10 group sizes:", group_train[:10])

X_train: (3966682, 168)
X_val: (991665, 168)
Number of train groups: 159836
Number of validation groups: 39959
First 10 group sizes: [28 32  5 21 28 29 33 34 16 28]


Custom NDCG@5 evaluation function

In [15]:
def dcg_at_k(relevances, k=5):
    """
    Computes DCG@k for one ranked list.
    """
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevances) + 2))
    gains = (2 ** relevances - 1)
    return np.sum(gains / discounts)


def ndcg_at_k_for_group(y_true, y_score, k=5):
    """
    Computes NDCG@k for one search group.
    """
    order = np.argsort(-y_score)
    ranked_relevance = y_true[order]

    ideal_order = np.argsort(-y_true)
    ideal_relevance = y_true[ideal_order]

    dcg = dcg_at_k(ranked_relevance, k=k)
    idcg = dcg_at_k(ideal_relevance, k=k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_ndcg_at_k(df, y_true_col, y_score_col, group_col="srch_id", k=5):
    """
    Computes mean NDCG@k over all searches.
    """

    scores = []

    for _, group in df.groupby(group_col):
        y_true = group[y_true_col].to_numpy()
        y_score = group[y_score_col].to_numpy()

        scores.append(ndcg_at_k_for_group(y_true, y_score, k=k))

    return np.mean(scores)

Simple heuristic baseline

In [16]:
# This baseline is not meant to be best.
# It gives a simple comparison score before training a real model.
val_baseline = val_feat[["srch_id", "prop_id", "relevance"]].copy()

val_baseline["baseline_score"] = (
    2.0 * val_feat["promotion_flag"]
    + 1.5 * val_feat["prop_location_score2"]
    + 1.0 * val_feat["prop_review_score"]
    + 0.5 * val_feat["prop_starrating"]
    - 1.0 * val_feat["price_ratio_to_search_mean"]
)

baseline_ndcg = mean_ndcg_at_k(
    val_baseline,
    y_true_col="relevance",
    y_score_col="baseline_score",
    group_col="srch_id",
    k=5
)

print("Baseline validation NDCG@5:", baseline_ndcg)

Baseline validation NDCG@5: 0.2704368737893543


Train LightGBM LambdaMART ranker

In [17]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[5],
    boosting_type="gbdt",

    # These parameters are reasonable starting values.
    # You can tune them later.
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

ranker.fit(
    X_train,
    y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    eval_at=[5],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.466348 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16597
[LightGBM] [Info] Number of data points in the train set: 3966682, number of used features: 168
Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@5: 0.379711
[100]	valid_0's ndcg@5: 0.388638
[150]	valid_0's ndcg@5: 0.394238
[200]	valid_0's ndcg@5: 0.398395
[250]	valid_0's ndcg@5: 0.4007
[300]	valid_0's ndcg@5: 0.401638
[350]	valid_0's ndcg@5: 0.402157
[400]	valid_0's ndcg@5: 0.403163
[450]	valid_0's ndcg@5: 0.403981
[500]	valid_0's ndcg@5: 0.404169
[550]	valid_0's ndcg@5: 0.404229
[600]	valid_0's ndcg@5: 0.404756
[650]	valid_0's ndcg@5: 0.405015
[700]	valid_0's ndcg@5: 0.404965
Early stopping, best iteration is:
[660]	valid_0's ndcg@5: 0.405228


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,800
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


Evaluate LightGBM ranker on validation set

In [18]:
val_predictions = ranker.predict(X_val, num_iteration=ranker.best_iteration_)

val_eval = val_feat[["srch_id", "prop_id", "relevance"]].copy()
val_eval["prediction"] = val_predictions

ranker_ndcg = mean_ndcg_at_k(
    val_eval,
    y_true_col="relevance",
    y_score_col="prediction",
    group_col="srch_id",
    k=5
)

print("LightGBM Ranker validation NDCG@5:", ranker_ndcg)
print("Baseline validation NDCG@5:", baseline_ndcg)

/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


LightGBM Ranker validation NDCG@5: 0.4052283250903438
Baseline validation NDCG@5: 0.2704368737893543


Inspect feature importance

In [19]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": ranker.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(40))

,feature,importance
10,prop_log_historical_price,1072
11,price_usd,1070
155,prop_id_mean_log_price,1030
141,location1_diff_from_search_mean,1028
159,prop_id_mean_location_score2,1021
153,prop_id_median_price,1013
151,prop_id_count,1012
9,prop_location_score2,1009
154,prop_id_std_price,901
142,location2_diff_from_search_mean,897


Prepare full training data and test data for final model

In [20]:
# For final Kaggle training, we fit preprocessing and aggregates on the full training set.
full_params = fit_preprocessing_params(train_raw)

# Prepare train/test without aggregate features first
full_train_no_agg = prepare_features(train_raw, full_params)
test_no_agg = prepare_features(test_raw, full_params)

# Fit aggregates on full training data only
full_prop_agg, full_dest_agg, full_prop_dest_agg = fit_aggregate_features(full_train_no_agg)

# Merge aggregates
full_train_feat = add_aggregate_features(full_train_no_agg, full_prop_agg, full_dest_agg, full_prop_dest_agg)
test_feat = add_aggregate_features(test_no_agg, full_prop_agg, full_dest_agg, full_prop_dest_agg)

full_train_feat = fill_aggregate_missing_values(full_train_feat)
test_feat = fill_aggregate_missing_values(test_feat)

print("Full train prepared shape:", full_train_feat.shape)
print("Test prepared shape:", test_feat.shape)

Full train prepared shape: (4958347, 176)
Test prepared shape: (4959183, 171)


Align final train/test feature columns

In [21]:
# Rebuild feature list from full training data.
final_feature_cols = [
    col for col in full_train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(full_train_feat[col])
]

# Ensure test has all final feature columns.
missing_in_test = [col for col in final_feature_cols if col not in test_feat.columns]
print("Missing features in test:", missing_in_test)

# If anything is missing, create it as 0.
# Normally this should not happen if the functions are applied consistently.
for col in missing_in_test:
    test_feat[col] = 0

# Keep only final features.
full_train_feat = full_train_feat.sort_values("srch_id").reset_index(drop=True)
test_feat = test_feat.sort_values("srch_id").reset_index(drop=True)

X_full = full_train_feat[final_feature_cols]
y_full = full_train_feat["relevance"].astype(int)
group_full = full_train_feat.groupby("srch_id").size().to_numpy()

X_test = test_feat[final_feature_cols]

print("X_full:", X_full.shape)
print("X_test:", X_test.shape)
print("Number of final features:", len(final_feature_cols))

Missing features in test: []
X_full: (4958347, 168)
X_test: (4959183, 168)
Number of final features: 168


Train final model on the full training data

In [22]:
# Use the best iteration from validation if available.
# If early stopping stopped at e.g. 350 trees, train final model with that many trees.
best_n_estimators = ranker.best_iteration_

if best_n_estimators is None or best_n_estimators <= 0:
    best_n_estimators = 800

print("Training final model with n_estimators =", best_n_estimators)

final_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[5],
    boosting_type="gbdt",

    n_estimators=best_n_estimators,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_ranker.fit(
    X_full,
    y_full,
    group=group_full
)

Training final model with n_estimators = 660


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.604407 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16641
[LightGBM] [Info] Number of data points in the train set: 4958347, number of used features: 168


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,660
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


Predict test scores and create Kaggle submission

In [23]:
test_scores = final_ranker.predict(X_test)

submission = test_feat[["srch_id", "prop_id"]].copy()
submission["score"] = test_scores

# Sort hotels within each search by predicted score descending.
submission = submission.sort_values(
    ["srch_id", "score"],
    ascending=[True, False]
)

# Required Kaggle format:
# SearchId,PropertyId
submission = submission.rename(columns={
    "srch_id": "SearchId",
    "prop_id": "PropertyId"
})

submission = submission[["SearchId", "PropertyId"]]

display(submission.head(30))

submission_path = "submission_lgbm_ranker.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print("Submission shape:", submission.shape)

/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


,SearchId,PropertyId
6,1,99484
24,1,54937
27,1,61934
20,1,28181
19,1,24194
21,1,34263
7,1,95031
22,1,5543
14,1,90385
23,1,50162


Saved submission to: submission_lgbm_ranker.csv
Submission shape: (4959183, 2)


Save final feature importance

In [24]:
final_feature_importance = pd.DataFrame({
    "feature": final_feature_cols,
    "importance": final_ranker.feature_importances_
}).sort_values("importance", ascending=False)

display(final_feature_importance.head(50))

final_feature_importance.to_csv("feature_importance_lgbm_ranker.csv", index=False)
print("Saved feature importance to feature_importance_lgbm_ranker.csv")

,feature,importance
10,prop_log_historical_price,1157
151,prop_id_count,1122
155,prop_id_mean_log_price,1073
11,price_usd,1063
141,location1_diff_from_search_mean,1020
159,prop_id_mean_location_score2,1007
153,prop_id_median_price,1005
9,prop_location_score2,996
142,location2_diff_from_search_mean,928
154,prop_id_std_price,926


Saved feature importance to feature_importance_lgbm_ranker.csv
